In [ ]:
from IPython.display import HTML, display

display(HTML("""
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
  mermaid.initialize({ startOnLoad: false, theme: "neutral", securityLevel: "strict" });
  const renderMermaid = async () => {
    const nodes = [...document.querySelectorAll(".mermaid:not([data-processed])")];
    if (nodes.length) await mermaid.run({ nodes });
  };
  new MutationObserver(() => renderMermaid()).observe(document.body, { childList: true, subtree: true });
  renderMermaid();
</script>
"""))


# AgentOps Lab 05 - Add state and memory with LangGraph

This notebook is where the AgentOps scenario benefits from a graph runtime. The investigation now has state, repeated evidence collection, confidence thresholds, and memory risk. LangGraph is pedagogically useful because it represents agent execution as state + nodes + edges, while distinguishing thread-scoped state from longer-term memory stores.



## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — State, graph execution, and memory

### Concepts to master

- state as the contract between nodes
- conditional routing and confidence loops
- thread-scoped state versus long-term memory

### Implementation walkthrough

`state_memory_langgraph.py` models triage, evidence collection, analysis, investigation loops, and recommendation. The notebook explains the LangGraph mapping while keeping the runnable path dependency-free.

### Deliberate failure case

Persist the stale memory `Checkout problems are usually caused by Redis` and rerun the incident. Watch how irrelevant memory can bias diagnosis away from current evidence.

### Learner exercise

Add a memory validator that stores preferences but blocks unverified incident-cause memories unless they include evidence IDs and expiry.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Incident state

The graph carries a typed state object through each node:

```python
class IncidentState(TypedDict):
    request: str
    service: str | None
    evidence: list[dict]
    suspected_cause: str | None
    confidence: float
    attempts: int
    recommendation: str | None
```

This is thread-scoped working state: it belongs to the current incident run. It should not automatically become long-term memory.


## Graph shape

<pre class="mermaid">
flowchart TD
    START([START]) --> T["triage"]
    T --> N{"need evidence?"}
    N -- "no" --> R["recommend"]
    N -- "yes" --> C["collect evidence"]
    C --> A["analyze"]
    A --> S{"confidence sufficient?"}
    S -- "no: investigate" --> C
    S -- "yes: finish" --> R
    R --> END([END])
</pre>

This is the same agent loop, but made explicit. Instead of an opaque `while` loop, each node has a job and each edge explains why the run moves forward, loops, or stops.


## Real LangGraph shape

The optional LangGraph implementation follows this structure:

```python
from langgraph.graph import START, END, StateGraph

builder = StateGraph(IncidentState)
builder.add_node("triage", triage)
builder.add_node("collect_evidence", collect_evidence)
builder.add_node("analyze", analyze)
builder.add_node("recommend", recommend)

builder.add_edge(START, "triage")
builder.add_conditional_edges("analyze", decide_next_step, {
    "investigate": "collect_evidence",
    "finish": "recommend",
})
builder.add_edge("recommend", END)
graph = builder.compile()
```

The executable lab below uses a dependency-free runner with the same state transitions so it works immediately in this repo. Install `requirements.txt` when you want to port it to real LangGraph.


In [ ]:
from pathlib import Path
import sys

repo_root = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / "curriculum" / "advanced" / "05-incident-response-capstone" / "agentops_lab").exists()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the repository checkout.")
sys.path.insert(0, str(repo_root / "curriculum" / "advanced" / "05-incident-response-capstone"))

from agentops_lab.state_memory_langgraph import (
    MemoryStore,
    collect_evidence,
    memory_bias_experiment,
    run_state_graph,
)


## Run the graph

The graph starts with triage, loops through evidence collection and analysis until confidence is sufficient, and then recommends a bounded response.


In [ ]:
state, path = run_state_graph("Acme European users report checkout failures.")
path


In [ ]:
print("suspected_cause:", state["suspected_cause"])
print("confidence:", state["confidence"])
print("attempts:", state["attempts"])
print("recommendation:", state["recommendation"])


## Short-term state versus long-term memory

Thread-scoped state is evidence for this incident. Long-term memory can influence future incidents, so it needs stricter controls: scope, validation, provenance, auditability, and reversibility.

First store a useful preference:

```json
{ "customer": "Acme", "preference": "Always prioritize fast resolution" }
```

Then accidentally store a dubious fact:

```json
{ "customer": "Acme", "fact": "Checkout problems are usually caused by Redis." }
```

Now run a new incident and observe whether stale memory biases diagnosis.


In [ ]:
experiment = memory_bias_experiment()
print("baseline:", experiment["baseline"]["state"]["suspected_cause"])
print("biased:", experiment["biased"]["state"]["suspected_cause"])
print("repaired:", experiment["repaired"]["state"]["suspected_cause"])


## What happened?

The baseline run diagnoses from evidence. The biased run trusts unverified long-term memory and shifts toward Redis. The repaired run deactivates the bad memory and returns to the evidence-backed suspected cause.

This is why long-term memory should not be treated as a bag of facts. A good memory system needs:

- scope by customer, tenant, user, and task;
- provenance for every write;
- validation before facts influence decisions;
- audit logs and inspection; and
- reversible deletion or deactivation.


## Exercises

- Add a `memory_policy(record)` function that blocks unverified operational facts from influencing diagnosis.
- Add a confidence penalty when evidence conflicts with memory.
- Port the dependency-free runner to real LangGraph using `StateGraph` and conditional edges.
- Add checkpoint persistence so the graph can resume after collecting deployments but before querying logs.

References: [LangGraph persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/), [LangGraph memory](https://langchain-ai.github.io/langgraph/concepts/memory/), [LangGraph low-level concepts](https://langchain-ai.github.io/langgraph/concepts/low_level/), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).
